# 09: Final Production Index Build

This notebook builds the production index using the best experimentally selected configuration. We extract the full set of passages, chunk them using the optimal strategy, embed them, and construct the final FAISS and BM25 indices. The indices and metadata are saved as artifacts for export.

**Note**: Do not commit large index files to Git!

In [ ]:
import sys
import os
import time
from pathlib import Path

# Find repository root and add to path
try:
    from colab.src.utils import find_repo_root
except ImportError:
    sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
    from colab.src.utils import find_repo_root

repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from colab.src.utils import load_config, set_seed, get_artifacts_dir, print_header
from colab.src.dataset_utils import load_msmarco_xi, extract_all_passages
from colab.src.chunking import run_chunking
from colab.src.embeddings import benchmark_models, EmbeddingModel
from colab.src.retrieval import FAISSIndex, BM25Index

# Set reproducibility seed
set_seed(42)

## 1. Load Configuration and Setup

We load the configuration which contains the optimal parameters determined in previous notebooks.

In [ ]:
config = load_config()
print_header("Experiment Configuration")
print(f"Target Language: {config.get('language', 'en')}")

best_chunking_strategy = config.get('best_chunk_strategy', 'semantic')
best_embedding_model = config.get('best_embedding_model', 'bge-m3')
index_type = config.get('index_type', 'HNSWFlat')

print(f"Selected Chunking: {best_chunking_strategy}")
print(f"Selected Embedding: {best_embedding_model}")
print(f"Selected Index Type: {index_type}")

## 2. Load Dataset and Chunking

Extract all available passages for the target language and apply the chosen chunking strategy.

In [ ]:
print_header("Loading Dataset")
lang = config.get('language', 'en')
dataset = load_msmarco_xi(lang, split='train')

passages = extract_all_passages(dataset, limit=None)  # Use all passages for production
print(f"Extracted {len(passages)} passages.")

print_header("Chunking Passages")
chunking_results = run_chunking(
    passages, 
    strategy=best_chunking_strategy, 
    chunk_size=config.get('chunk_size', 512),
    chunk_overlap=config.get('chunk_overlap', 50)
)

chunks = chunking_results.chunks
print(f"Generated {len(chunks)} chunks from {len(passages)} passages.")

## 3. Embedding the Chunks

Generate vector embeddings for all extracted chunks using the optimal model.

In [ ]:
print_header(f"Embedding Chunks with {best_embedding_model}")

embedder = EmbeddingModel(model_name=best_embedding_model)
texts_to_embed = [c.text for c in chunks]

start_time = time.time()
embeddings = embedder.embed_documents(texts_to_embed)
embed_time = time.time() - start_time
print(f"Embedding completed in {embed_time:.2f} seconds.")
print(f"Embedding shape: {embeddings.shape}")

## 4. Build Production Indices

Construct both the dense (FAISS) and sparse (BM25) indices for hybrid retrieval.

In [ ]:
print_header("Building FAISS Index")
dimension = embeddings.shape[1]
faiss_index = FAISSIndex(dimension=dimension, index_type=index_type)
faiss_index.build(embeddings, [c.metadata for c in chunks])
print(f"FAISS Index built. Total vectors: {faiss_index.ntotal}")

print_header("Building BM25 Index")
bm25_index = BM25Index()
bm25_index.build(texts_to_embed, [c.metadata for c in chunks])
print("BM25 Index built successfully.")

## 5. Verify and Save Artifacts

Perform a sample search to verify the index works, then save the indices to disk.

In [ ]:
print_header("Index Verification")
sample_query = "What is the capital of India?"
sample_query_emb = embedder.embed_queries([sample_query])[0]

results = faiss_index.search(sample_query_emb, k=3)
print(f"Sample Query: '{sample_query}'")
for i, res in enumerate(results):
    print(f"Result {i+1} (Score: {res['score']:.4f}): {res['metadata'].get('doc_id', 'Unknown')}")

print_header("Saving Indices")
final_index_dir = get_artifacts_dir() / "final_index"
final_index_dir.mkdir(parents=True, exist_ok=True)

faiss_path = final_index_dir / "faiss.index"
faiss_index.save(str(faiss_path))
bm25_path = final_index_dir / "bm25.pkl"
bm25_index.save(str(bm25_path))

from colab.src.utils import save_json
chunk_meta_path = final_index_dir / "chunk_metadata.json"
save_json([c.metadata for c in chunks], chunk_meta_path)

faiss_size = os.path.getsize(faiss_path) / (1024 * 1024) if faiss_path.exists() else 0
bm25_size = os.path.getsize(bm25_path) / (1024 * 1024) if bm25_path.exists() else 0

print_header("Final Index Statistics")
print(f"Number of vectors: {faiss_index.ntotal}")
print(f"Vector dimension: {dimension}")
print(f"Index type: {index_type}")
print(f"FAISS Index Size: {faiss_size:.2f} MB")
print(f"BM25 Index Size: {bm25_size:.2f} MB")
print("\nWARNING: Do not commit these large index files to Git!")